# StarGAN Blur-to-Sharp + Attribute Translation (Kaggle)

This notebook trains and evaluates the custom StarGAN variant with:
- blurred input -> sharp + attribute-edited output
- identity constraint (FaceNet)
- perceptual sharpness loss (VGG)
- W&B metrics + checkpoint testing

In [ ]:
!pip install -q facenet-pytorch wandb

In [ ]:
import os
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if torch.cuda.is_available():
    print('VRAM(GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

## Optional: W&B login

In [ ]:
# import wandb
# wandb.login(key='YOUR_WANDB_KEY')

In [ ]:
from config import Config
from trainer import Trainer

## Configure experiment

In [ ]:
cfg = Config()

# Runtime knobs for Kaggle 16GB GPU
cfg.batch_size = 16
cfg.num_epochs = 30
cfg.use_amp = True

# Validation/test speed-vs-quality tradeoff
cfg.val_max_batches = 120
cfg.test_max_batches = 200

# W&B
cfg.use_wandb = True
cfg.wandb_project = 'stargan-blur-upscale'
cfg.wandb_run_name = 'kaggle-run-01'

# Print core config
print('image_size:', cfg.image_size)
print('batch_size:', cfg.batch_size)
print('epochs:', cfg.num_epochs)
print('dataset root:', cfg.celeba_root)

## Train

In [ ]:
trainer = Trainer(cfg)
trainer.train()

## Test Any Checkpoint

In [ ]:
# Example: evaluate epoch checkpoint
ckpt_path = '/kaggle/working/checkpoints/ckpt_ep10.pth'

cfg_eval = Config()
cfg_eval.use_wandb = False
trainer_eval = Trainer(cfg_eval)
metrics = trainer_eval.evaluate_test_checkpoint(ckpt_path)
metrics

## Compare Multiple Checkpoints

In [ ]:
import glob

cfg_cmp = Config()
cfg_cmp.use_wandb = False
cmp_trainer = Trainer(cfg_cmp)

ckpts = sorted(glob.glob('/kaggle/working/checkpoints/ckpt_ep*.pth'))
summary = []
for p in ckpts[-5:]:
    m = cmp_trainer.evaluate_test_checkpoint(p)
    summary.append((os.path.basename(p), m['test/psnr'], m['test/ssim'], m['test/id'], m['test/perc']))

summary

## Visualize Latest Samples

In [ ]:
from IPython.display import Image, display

sample_files = sorted(os.listdir('/kaggle/working/samples'))
print('num sample grids:', len(sample_files))
if sample_files:
    display(Image(filename=os.path.join('/kaggle/working/samples', sample_files[-1])))